
## 1. Install Dependencies

Installs the packages required to run Selenium with Chrome in Google Colab and generate browser user agents for Goodreads requests.



In [ ]:
!pip install google-colab-selenium
!pip install fake-useragent

## 2. Browser Setup

Imports the required libraries and configures a headless Chrome browser for scraping Goodreads pages within Google Colab.

In [ ]:
from selenium import webdriver
from google_colab_selenium import Chrome
from fake_useragent import UserAgent
from google.colab import userdata
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.common.by import By

def web_driver():
    ua = UserAgent()
    user_agent = ua.chrome

    options = webdriver.ChromeOptions()
    options.add_argument(f"user-agent={user_agent}")

    return Chrome(options=options)

browser = web_driver()

## 3. Notion Configuration

Loads the Notion API token securely from Google Colab Secrets and defines the database connections used to update books and authors.

In [ ]:
NOTION_TOKEN = userdata.get("NOTION_TOKEN")

BOOK_DATABASE_ID = "1f89bcd8d884472eb01a6af004b0dff2"
AUTHOR_DATABASE_ID = "1afd46f033614842b1e7b7bab00b7ff2"

NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28"
}

TIMEOUT_IN_SECONDS = 20

## 4. Notion API Helpers

Provides reusable functions for querying databases and creating or updating pages through the Notion API.

In [ ]:
import requests
import dateutil.parser as dparser

from bs4 import BeautifulSoup
from urllib.parse import urlparse

In [ ]:
def read_database(database_id, read_data):
    url = f"https://api.notion.com/v1/databases/{database_id}/query"

    response = requests.post(
        url,
        headers=NOTION_HEADERS,
        json=read_data
    )
    response.raise_for_status()

    return response.json()

In [ ]:
def create_page(new_page_data):
    url = "https://api.notion.com/v1/pages"

    response = requests.post(
        url,
        headers=NOTION_HEADERS,
        json=new_page_data
    )
    response.raise_for_status()

    return response.json()

In [ ]:
def update_page(page_id, update_data):
    url = f"https://api.notion.com/v1/pages/{page_id}"

    response = requests.patch(
        url,
        headers=NOTION_HEADERS,
        json=update_data
    )
    response.raise_for_status()

    return response.json()

## 5. Author Management

Checks the Authors database for each scraped author and creates a new author record when one does not already exist.

In [ ]:
def find_author(name):
    print(f"Checking for author: {name}")

    read_data = {
        "filter": {
            "property": "Name",
            "title": {
                "equals": name
            }
        }
    }

    results = read_database(AUTHOR_DATABASE_ID, read_data).get("results", [])

    if results:
        print(f"Author found: {name}")
        return results[0].get("id")

    print(f"Author not found: {name}")
    return None

In [ ]:
def add_author(name):
    new_page_data = {
        "parent": {
            "database_id": AUTHOR_DATABASE_ID
        },
        "properties": {
            "Name": {
                "title": [
                    {
                        "text": {
                            "content": name
                        }
                    }
                ]
            }
        }
    }

    new_author = create_page(new_page_data)
    print(f"Author added: {name}")

    return new_author.get("id")

## 6. Build Notion Book Properties

Converts scraped Goodreads metadata into the property structure used by the Notion Books database. Authors are matched to existing records or added to the Authors database when necessary.

In [ ]:
def get_book_properties(book):
    author_ids = []

    for author_name in book.get("author_list"):
        author_id = find_author(author_name)

        if author_id is not None:
            author_ids.append({"id": author_id})
        else:
            author_ids.append({"id": add_author(author_name)})

    properties = {
        "Name": {
            "title": [
                {
                    "type": "text",
                    "text": {
                        "content": book.get("name")
                    }
                }
            ]
        },
        "Goodreads": {
            "url": book.get("goodreads")
        },
        "Genre": {
            "multi_select": book.get("genre_list")
        },
        "Audience": {
            "select": {
                "name": book.get("audience")
            }
        },
        "Fic Type": {
            "select": {
                "name": book.get("fic_type")
            }
        }
    }

    if author_ids:
        properties["Author"] = {
            "relation": author_ids
        }

    if book.get("pub_date") is not None:
        properties["Pub Date"] = {
            "date": {
                "start": book.get("pub_date").strftime("%Y-%m-%d")
            }
        }

    return properties

## 7. Scrape Goodreads Metadata

Extracts book metadata from a Goodreads book page, including title, author, publication date, genres, audience, and fiction/nonfiction classification.

Some Goodreads genres are excluded or normalized to better match the categories used in the Notion library.

In [ ]:
def get_book_info(soup, url):
    print("Getting book info")

    # Title
    title = soup.find_all("h1", class_="Text__title1")[0].text
    print(f"Title: {title}")
    print()

    # Authors
    author_elements = soup.find_all("span", class_="ContributorLink__name")
    author_list = []

    print("Authors:")
    for author in author_elements:
        if author.text not in author_list:
            author_list.append(author.text)
            print(author.text)
    print()

    # Publication date
    try:
        featured_details = soup.find_all("div", class_="FeaturedDetails")[0].text

        # Remove page count before parsing the publication date.
        date_text = featured_details
        if "pages" in featured_details:
            date_text = featured_details.split("pages")[1]

        pub_date = dparser.parse(date_text, fuzzy=True)
    except (IndexError, ValueError, TypeError):
        pub_date = None

    # Genre, audience, and fiction type
    genre_elements = soup.find_all(
        "span",
        class_="BookPageMetadataSection__genreButton"
    )

    genre_list = []
    fic_type = "Fiction"
    audience = "Adult"
    audience_found = False

    ignore_genres = [
        "Young Adult Fantasy",
        "New Adult",
        "Fantasy Romance",
        "Magic",
        "Dc Comics",
        "Comic Book",
        "Science Fiction Fantasy",
        "Mystery Thriller",
        "Christmas",
        "Audiobook",
        "Historical Mystery",
        "High Fantasy",
        "Epic Fantasy"
    ]

    audience_options = [
        "Adult",
        "Young Adult",
        "Middle Grade",
        "Children"
    ]

    print("Genres:")

    for genre in genre_elements:
        genre_element = genre.find("span", class_="Button__labelItem")
        genre_text = genre_element.text
        should_add = True

        if genre_text in ignore_genres:
            should_add = False
        else:
            # Determine fiction type.
            if genre_text == "Nonfiction":
                fic_type = "Non-Fiction"
                should_add = False
            elif genre_text == "Fiction":
                should_add = False

            # Determine audience.
            if not audience_found and genre_text in audience_options:
                audience = genre_text
                audience_found = True
                should_add = False
            elif genre_text in audience_options:
                should_add = False

            # Normalize Goodreads genres to the categories used in Notion.
            if genre_text in ["Graphic Novels", "Comics", "Graphic Novels Comics"]:
                genre_text = "graphic novel"
                should_add = True

            if genre_text in ["LGBT", "Queer"]:
                genre_text = "lgbtqia"

            if genre_text == "Lesbian":
                genre_text = "sapphic"

        if should_add:
            normalized_genre = genre_text.lower()

            genre_list.append({
                "name": normalized_genre
            })

            print(normalized_genre)

    print()
    print(f"Fic Type: {fic_type}")
    print()
    print(f"Audience Type: {audience}")

    book = {
        "name": title,
        "goodreads": url,
        "genre_list": genre_list,
        "audience": audience,
        "fic_type": fic_type,
        "pub_date": pub_date,
        "author_list": author_list
    }

    return book

## 8. Load Goodreads Pages

Uses the configured Chrome browser to load each Goodreads page and waits for the book title to appear before parsing the page with BeautifulSoup.

In [ ]:
def load_goodreads_page(url):
    print(f"Loading Goodreads page: {url}")

    browser.get(url)
    print("Waiting for Goodreads content...")

    WebDriverWait(browser, TIMEOUT_IN_SECONDS).until(
        ec.presence_of_element_located(
            (By.CLASS_NAME, "Text__title1")
        )
    )
    print("Goodreads page loaded")

    html = browser.page_source
    print("HTML retrieved")

    soup = BeautifulSoup(html, "html.parser")
    print("Page parsed")

    book = get_book_info(soup, url)

    return book

In [ ]:
def is_url(url):
  try:
    result = urlparse(url)
    return all([result.scheme, result.netloc])
  except ValueError:
    return False

## 9. Update Books in Notion

Finds books that still need metadata, validates their Goodreads URLs, scrapes the available book information, and updates the existing records in Notion.

In [ ]:
def update_book_info(book, book_id):
    print(f"Updating Notion: {book.get('name')}")

    properties = get_book_properties(book)

    update_data = {
        "properties": properties
    }

    update_page(book_id, update_data)

    print(f"Updated successfully: {book.get('name')}")
    print()

In [ ]:
def pull_books_needing_info():
    print("Finding books that need metadata...")

    read_data = {
        "filter": {
            "property": "Audience",
            "select": {
                "is_empty": True
            }
        }
    }

    results = read_database(
        BOOK_DATABASE_ID,
        read_data
    ).get("results", [])

    print(f"Found {len(results)} book(s) to process")
    print()

    for book_record in results:
        try:
            goodreads_url = book_record.get("properties").get("Goodreads").get("url")
            book_id = book_record.get("id")

            if is_url(goodreads_url):
                print(f"Valid Goodreads URL: {goodreads_url}")

                book = load_goodreads_page(goodreads_url)
                update_book_info(book, book_id)

            else:
                print(f"Invalid or missing Goodreads URL: {goodreads_url}")
                print()

        except Exception as e:
            print(f"Error processing book: {e}")
            print()

In [ ]:
pull_books_needing_info()